Grouped Query Attention / GQA


In [47]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00


In [48]:
import torch

In [56]:
import torch
import math

def gqa_forward(Q, K, V):
    """
    Grouped Query Attention forward pass, no mask.

    Args:
        Q: tensor [B, Hq,  Tq, D]
        K: tensor [B, Hkv, Tk, D]
        V: tensor [B, Hkv, Tk, D]
        Hq % Hkv == 0. Query head h attends to kv head h // (Hq // Hkv).

    Returns:
        tensor [B, Hq, Tq, D], dtype float64

    TODO:
      1. Coerce with torch.as_tensor(..., dtype=torch.float64). Read Hq and Hkv
         off the shapes, set g = Hq // Hkv.
      2. Make each query head see its kv head. Options: repeat_interleave on
         dim 1, or unsqueeze + expand + reshape, or reshape Q to
         [B, Hkv, g, Tq, D] and broadcast K/V against it.
         Whichever you pick, verify the mapping is h // g and not h % Hkv.
         Tiling and interleaving are different orderings and both type-check.
      3. scores = Q @ K^T / sqrt(D). Use transpose(-2, -1) or einsum. Shape
         should be [B, Hq, Tq, Tk].
      4. weights = softmax over the KEY axis. That is the last dim, not Tq.
      5. output = weights @ V_expanded, shape back to [B, Hq, Tq, D].

    No Python loop over B, heads, or positions. Keep ops out-of-place so this
    stays differentiable w.r.t. Q.
    """
    B, H, To, D = Q.shape
    _, R, Ti, _ = K.shape
    q = torch.as_tensor(Q, dtype=torch.float64)
    k = torch.as_tensor(K, dtype=torch.float64)
    v = torch.as_tensor(V, dtype=torch.float64)
    G = H // R
    q = q.view(B, R, G, To, D)
    v = v.view(B, R, Ti, D)
    k = k.view(B, R, Ti, D)
    logits = torch.einsum('brgod,brid->brgoi', q, k) * math.sqrt(D) ** -1.0
    attn = torch.softmax(logits, dim=-1)
    output = torch.einsum('brgoi,brid->brgod', attn, v)
    output = output.view(B, H, To, D)
    return output
    raise NotImplementedError


def solve(input_str):
    toks = input_str.split()
    B, Hq, Hkv, Tq, Tk, D = (int(t) for t in toks[:6])

    nQ = B * Hq * Tq * D
    nKV = B * Hkv * Tk * D
    nums = [float(t) for t in toks[6:]]
    assert len(nums) == nQ + 2 * nKV, \
        f"expected {nQ + 2*nKV} values after the header, got {len(nums)}"

    i = 0
    Q = torch.tensor(nums[i:i + nQ], dtype=torch.float64).reshape(B, Hq, Tq, D)
    i += nQ
    K = torch.tensor(nums[i:i + nKV], dtype=torch.float64).reshape(B, Hkv, Tk, D)
    i += nKV
    V = torch.tensor(nums[i:i + nKV], dtype=torch.float64).reshape(B, Hkv, Tk, D)

    out = gqa_forward(Q, K, V)
    return " ".join(f"{v:.6f}" for v in out.flatten().tolist())

In [57]:
import sys, time, torch

TOL = 5e-7

# ---- hardcoded cases: (name, stdin_text, expected flat output) ----
_ordering_in = (
    "2 2 1 2 1 2\n"
    + " ".join(["1"] * 16) + "\n"      # Q, all ones (irrelevant: Tk=1)
    + " ".join(["0"] * 4) + "\n"       # K, all zeros
    + "1 2 3 4"                        # V: batch0 kv0 = [1,2], batch1 kv0 = [3,4]
)

TESTS = [
    ("example 1 (Hq=2, Hkv=1, shared kv head)",
     "1 2 1 1 1 1\n1 2\n3\n4",
     [4.0, 4.0]),

    ("example 2 (single everything)",
     "1 1 1 1 1 1\n1\n0\n5",
     [5.0]),

    ("equal scores -> uniform weights -> mean of V",
     "1 1 1 1 2 1\n0\n1 2\n2 4",
     [3.0]),

    # scores are [4,0] before scaling, [2,0] after. Fails if you skip /sqrt(D).
    ("1/sqrt(D) scaling, D=4",
     "1 1 1 1 2 4\n1 1 1 1\n1 1 1 1 0 0 0 0\n1 0 0 0 0 1 0 0",
     [0.88079708, 0.11920292, 0.0, 0.0]),

    # h//g gives [10,10,20,20]; h%Hkv gives [10,20,10,20].
    ("head -> kv mapping (Hq=4, Hkv=2)",
     "1 4 2 1 1 1\n1 1 1 1\n0 0\n10 20",
     [10.0, 10.0, 20.0, 20.0]),

    # softmax over dim=-2 instead of -1 yields [0.5, 0.5] here.
    ("per-query-position weights (Tq=2, Tk=2)",
     "1 1 1 2 2 1\n1 0\n1 0\n0 1",
     [0.26894142, 0.5]),

    # scores are [1e6, 0]; naive exp() overflows to inf -> nan.
    ("numerical stability, huge scores",
     "1 1 1 1 2 1\n1000\n1000 0\n7 9",
     [7.0]),

    # head0: score gap ~141 -> one-hot on V row 0. head1: flat -> mean of V rows.
    ("Hq == Hkv, per-head independence, D=2",
     "1 2 2 1 2 2\n100 100 0 0\n1 1 0 0 0 0 0 0\n1 2 9 9 0 0 4 6",
     [1.0, 2.0, 2.0, 3.0]),

    ("batch + head + position flatten order",
     _ordering_in,
     [1, 2, 1, 2, 1, 2, 1, 2, 3, 4, 3, 4, 3, 4, 3, 4]),
]


def _check(name, thunk, expected):
    try:
        line = thunk().strip()
    except Exception as e:
        print(f"[FAIL] {name}\n       raised {type(e).__name__}: {e}")
        return False
    try:
        got = [float(t) for t in line.split()]
    except ValueError:
        print(f"[FAIL] {name}\n       unparseable: {line!r}")
        return False
    if len(got) != len(expected):
        print(f"[FAIL] {name}\n       expected {len(expected)} values, got {len(got)}")
        print(f"       got: {line[:120]}")
        return False
    bad = [i for i, (g, e) in enumerate(zip(got, expected)) if abs(g - e) > TOL]
    if bad:
        print(f"[FAIL] {name}")
        print(f"       expected: {' '.join(f'{v:.6f}' for v in expected)}")
        print(f"       got     : {' '.join(f'{v:.6f}' for v in got)}")
        print(f"       first mismatch at index {bad[0]}")
        return False
    print(f"[PASS] {name}")
    return True


def _prop(name, fn):
    try:
        ok, detail = fn()
    except Exception as e:
        print(f"[FAIL] {name}\n       raised {type(e).__name__}: {e}")
        return False
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    if not ok:
        print(f"       {detail}")
    return ok


# ---- property checks on random data, no reference solution needed ----

def _p_shape_dtype():
    B, Hq, Hkv, Tq, Tk, D = 2, 6, 3, 4, 5, 3
    out = gqa_forward(torch.randn(B, Hq, Tq, D, dtype=torch.float64),
                      torch.randn(B, Hkv, Tk, D, dtype=torch.float64),
                      torch.randn(B, Hkv, Tk, D, dtype=torch.float64))
    ok = (isinstance(out, torch.Tensor) and tuple(out.shape) == (B, Hq, Tq, D)
          and out.dtype == torch.float64)
    return ok, f"got {type(out)} {getattr(out, 'shape', None)} {getattr(out, 'dtype', None)}"


def _p_grouping():
    # If every key position of a kv head carries the same V row, then any head
    # mapped to that kv head must output exactly that row, whatever Q and K are.
    B, Hq, Hkv, Tq, Tk, D = 2, 6, 3, 4, 5, 3
    g = Hq // Hkv
    torch.manual_seed(0)
    Q = torch.randn(B, Hq, Tq, D, dtype=torch.float64)
    K = torch.randn(B, Hkv, Tk, D, dtype=torch.float64)
    base = torch.randn(B, Hkv, 1, D, dtype=torch.float64)
    V = base.expand(B, Hkv, Tk, D).contiguous()
    out = gqa_forward(Q, K, V)
    want = torch.empty(B, Hq, Tq, D, dtype=torch.float64)
    for h in range(Hq):
        want[:, h] = base[:, h // g]
    return torch.allclose(out, want, atol=1e-10), f"max err {(out - want).abs().max().item():.3e}"


def _p_key_permutation():
    # Attention is invariant to permuting the key axis of K and V together.
    B, Hq, Hkv, Tq, Tk, D = 2, 4, 2, 3, 7, 5
    torch.manual_seed(1)
    Q = torch.randn(B, Hq, Tq, D, dtype=torch.float64)
    K = torch.randn(B, Hkv, Tk, D, dtype=torch.float64)
    V = torch.randn(B, Hkv, Tk, D, dtype=torch.float64)
    perm = torch.randperm(Tk)
    a = gqa_forward(Q, K, V)
    b = gqa_forward(Q, K[:, :, perm], V[:, :, perm])
    return torch.allclose(a, b, atol=1e-10), f"max err {(a - b).abs().max().item():.3e}"


def _p_convex_hull():
    # Weights are nonnegative and sum to 1, so each output lies within the
    # per-dimension min/max of the V rows of its kv head.
    B, Hq, Hkv, Tq, Tk, D = 2, 6, 2, 3, 8, 4
    g = Hq // Hkv
    torch.manual_seed(2)
    Q = torch.randn(B, Hq, Tq, D, dtype=torch.float64) * 3
    K = torch.randn(B, Hkv, Tk, D, dtype=torch.float64) * 3
    V = torch.randn(B, Hkv, Tk, D, dtype=torch.float64)
    out = gqa_forward(Q, K, V)
    lo, hi = V.min(dim=2).values, V.max(dim=2).values
    worst = 0.0
    for h in range(Hq):
        kv = h // g
        worst = max(worst,
                    (lo[:, kv].unsqueeze(1) - out[:, h]).clamp(min=0).max().item(),
                    (out[:, h] - hi[:, kv].unsqueeze(1)).clamp(min=0).max().item())
    return worst < 1e-12, f"worst violation {worst:.3e}"


def _p_autograd():
    B, Hq, Hkv, Tq, Tk, D = 1, 4, 2, 3, 5, 4
    torch.manual_seed(3)
    Q = torch.randn(B, Hq, Tq, D, dtype=torch.float64, requires_grad=True)
    K = torch.randn(B, Hkv, Tk, D, dtype=torch.float64)
    V = torch.randn(B, Hkv, Tk, D, dtype=torch.float64)
    gqa_forward(Q, K, V).sum().backward()
    ok = (Q.grad is not None and Q.grad.shape == Q.shape
          and torch.isfinite(Q.grad).all().item() and Q.grad.abs().sum().item() > 0)
    return ok, f"grad={None if Q.grad is None else Q.grad.abs().sum().item()}"


R = [_check(n, (lambda s=s: solve(s)), e) for n, s, e in TESTS]
R.append(_prop("returns float64 tensor [B, Hq, Tq, D]", _p_shape_dtype))
R.append(_prop("constant-V per kv head pins the grouping", _p_grouping))
R.append(_prop("invariant to permuting the key axis", _p_key_permutation))
R.append(_prop("output inside convex hull of its V rows", _p_convex_hull))
R.append(_prop("autograd flows to Q", _p_autograd))
print(f"\n{sum(R)}/{len(R)} passed")


def run_timing():
    B, Hq, Hkv, Tq, Tk, D = 2, 32, 8, 256, 256, 64
    Q = torch.randn(B, Hq, Tq, D, dtype=torch.float64)
    K = torch.randn(B, Hkv, Tk, D, dtype=torch.float64)
    V = torch.randn(B, Hkv, Tk, D, dtype=torch.float64)
    t0 = time.perf_counter()
    out = gqa_forward(Q, K, V)
    dt = time.perf_counter() - t0
    print(f"\n[{B},{Hq},{Hkv},{Tq},{Tk},{D}] in {dt:.3f}s, out {tuple(out.shape)}")
    if dt > 2.0:
        print("  ^ slow; check for a Python loop over batch, heads, or positions")

def main():
    print(solve(sys.stdin.read()))

[PASS] example 1 (Hq=2, Hkv=1, shared kv head)
[PASS] example 2 (single everything)
[PASS] equal scores -> uniform weights -> mean of V
[PASS] 1/sqrt(D) scaling, D=4
[PASS] head -> kv mapping (Hq=4, Hkv=2)
[PASS] per-query-position weights (Tq=2, Tk=2)
[PASS] numerical stability, huge scores
[PASS] Hq == Hkv, per-head independence, D=2
[PASS] batch + head + position flatten order
[PASS] returns float64 tensor [B, Hq, Tq, D]
[PASS] constant-V per kv head pins the grouping
[PASS] invariant to permuting the key axis
[PASS] output inside convex hull of its V rows
[PASS] autograd flows to Q

14/14 passed
